In [1]:

# Task 3: SVM Hyperparameter Tuning
# Using GridSearchCV and RandomizedSearchCV

import time
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from scipy.stats import loguniform

# 1. Load Dataset
data = load_iris()
X = data.data
y = data.target

# 2. Split Dataset into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. Create Pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

# 4. Define Hyperparameter Grid
param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__kernel": ["linear", "rbf", "poly"],
    "svm__gamma": ["scale", "auto", 0.01, 0.1, 1]
}

# 5. GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

start_time = time.perf_counter()
grid_search.fit(X_train, y_train)
grid_time = time.perf_counter() - start_time

# 6. RandomizedSearchCV
param_dist = {
    "svm__C": loguniform(0.01, 1000),
    "svm__kernel": ["linear", "rbf", "poly"],
    "svm__gamma": loguniform(0.001, 10)
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

start_time = time.perf_counter()
random_search.fit(X_train, y_train)
random_time = time.perf_counter() - start_time

# 7. Display Best Results
print("===== GridSearchCV Results =====")
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)
print("Runtime:", round(grid_time, 4), "seconds")
print("Test Accuracy:", grid_search.score(X_test, y_test))

print("\n===== RandomizedSearchCV Results =====")
print("Best Parameters:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)
print("Runtime:", round(random_time, 4), "seconds")
print("Test Accuracy:", random_search.score(X_test, y_test))

# 8. Compare Both Methods
results = pd.DataFrame({
    "Method": ["GridSearchCV", "RandomizedSearchCV"],
    "Best CV Score": [
        grid_search.best_score_,
        random_search.best_score_
    ],
    "Runtime (seconds)": [grid_time, random_time],
    "Test Accuracy": [
        grid_search.score(X_test, y_test),
        random_search.score(X_test, y_test)
    ]
})

print("\n===== Comparison Table =====")
print(results.to_string(index=False))

# 9. Identify the Faster Method
if grid_time < random_time:
    print("\nFaster Method: GridSearchCV")
elif random_time < grid_time:
    print("\nFaster Method: RandomizedSearchCV")
else:
    print("\nBoth methods took approximately the same time.")


===== GridSearchCV Results =====
Best Parameters: {'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Best CV Score: 0.975
Runtime: 8.1433 seconds
Test Accuracy: 0.9333333333333333

===== RandomizedSearchCV Results =====
Best Parameters: {'svm__C': np.float64(0.1152644954031561), 'svm__gamma': np.float64(0.005337032762603957), 'svm__kernel': 'linear'}
Best CV Score: 0.975
Runtime: 0.2448 seconds
Test Accuracy: 0.9333333333333333

===== Comparison Table =====
            Method  Best CV Score  Runtime (seconds)  Test Accuracy
      GridSearchCV          0.975           8.143279       0.933333
RandomizedSearchCV          0.975           0.244787       0.933333

Faster Method: RandomizedSearchCV
